# HET Quick-Look Evidence & Presentation Playground

This notebook is intentionally **not another reduction pipeline**. It is a place to:

1. discover and inspect real VIRUS/LRS2 observations;
2. run the already-established quick-look workflows;
3. inspect the evidence carried by each result;
4. iterate rapidly on plots and presentation choices.

Current scientific defaults encoded by the package:

| Setting | VIRUS | LRS2 |
|---|---:|---:|
| Fractional extraction width | 5 detector pixels | 5 detector pixels |
| Collapse window | central 200 columns | central 200 columns |
| Collapse statistic | median | median |
| Gaussian splat FWHM | 1.5 arcsec | 1.2 arcsec |
| Pixel scale | 1.0 arcsec/pixel | 0.4 arcsec/pixel |
| Standard-star fiducial | (0, 0) in selected IFU | (0, 0) in LRS2 IFU |

The guiding philosophy is **evidence supply**: show the observer the data, topology, support, and pointing information clearly. Do not turn these plots into automated pass/fail decisions.

## 1. Session configuration

Edit only this cell for the dataset you want to explore.

The documented local examples use separate VIRUS and LRS2 roots, so the notebook keeps one root per instrument. `TRACE_ROOT` should be the directory that contains `Fiber_Locations/`.

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys

INSTRUMENT = "lrs2"           # "lrs2" or "virus"
DATES = {"lrs2": "20260512", "virus": "20260609"}
DATE = DATES[INSTRUMENT]

RAW_ROOTS = {
    "lrs2": Path("~/data/LRS2").expanduser(),
    "virus": Path("~/data/VIRUS").expanduser(),
}


def find_checkout_root():
    """Find the HET_LV_Tools checkout from any checkout subdirectory."""

    starts = []
    configured_root = os.environ.get("HET_LV_TOOLS_ROOT")
    if configured_root:
        starts.append(Path(configured_root).expanduser())
    starts.append(Path.cwd())
    starts.extend(Path(entry or ".").expanduser() for entry in sys.path)
    package_spec = importlib.util.find_spec("hetquicklook")
    if package_spec is not None and package_spec.origin:
        starts.append(Path(package_spec.origin))

    seen = set()
    for start in starts:
        start = start.resolve()
        if not start.is_dir():
            start = start.parent
        for candidate in (start, *start.parents):
            if candidate in seen:
                continue
            seen.add(candidate)
            if (candidate / "src" / "hetquicklook").is_dir():
                return candidate
    raise FileNotFoundError(
        "Could not locate the HET_LV_Tools checkout. "
        "Start Jupyter below the checkout or set HET_LV_TOOLS_ROOT."
    )


PROJECT_ROOT = find_checkout_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Directory containing Fiber_Locations/.
TRACE_ROOT = PROJECT_ROOT
RAW_ROOT = RAW_ROOTS[INSTRUMENT]

print("instrument:", INSTRUMENT)
print("date:", DATE)
print("raw root:", RAW_ROOT)
print("trace root:", TRACE_ROOT)


## 2. Imports and API inspection

The public quick-look path is still evolving at the workflow-integration boundary. This cell deliberately prints signatures so the notebook remains useful while we refine that API rather than hiding changes behind notebook-specific wrappers.

In [ ]:
from IPython.display import HTML, display
import inspect
import numpy as np
import matplotlib.pyplot as plt

from hetquicklook import (
    QuicklookConfig,
    discover_observations,
    load_observation,
)
import hetquicklook.workflows as ql_workflows
import hetquicklook.visualization as ql_visualization
import hetquicklook.topology as ql_topology
import hetquicklook.algorithms.detector as ql_detector
import hetquicklook.algorithms.trace as ql_trace
import hetquicklook.algorithms.extraction as ql_extraction

def show_signature(module, name):
    obj = getattr(module, name, None)
    if obj is None:
        print(f"{module.__name__}.{name}: not present")
    else:
        print(f"{module.__name__}.{name}{inspect.signature(obj)}")

for module, name in [
    (ql_workflows, "run_ldls_flat_quicklook"),
    (ql_workflows, "run_standard_star_quicklook"),
    (ql_workflows, "combine_lrs2_channel_products"),
    (ql_workflows, "combine_lrs2_channels"),
    (ql_workflows, "build_amplifier_topology"),
    (ql_workflows, "run_lrs2_channel_quicklooks"),
    (ql_detector, "reduce_amplifier_array"),
    (ql_trace, "fit_fiber_traces"),
    (ql_extraction, "extract_fractional_aperture"),
]:
    show_signature(module, name)

## 3. Discover a night and inspect observations

Discovery answers **what is present**, not whether an observation is scientifically complete.

In [ ]:
config = QuicklookConfig(RAW_ROOT)

discovered = discover_observations(
    config,
    instrument=INSTRUMENT,
    date=DATE,
)

print(f"Discovered {len(discovered)} observation archive(s)")
for i, item in enumerate(discovered):
    print(i, item)

Choose one archive to inspect. Multi-exposure tars remain multi-exposure
observations; the package exposes the encoded exposure groups separately.
The next section presents the scientific identity of each exposure in a
readable table before detector pixels are loaded.


In [ ]:
OBS_INDEX = 0
observation = load_observation(discovered[OBS_INDEX])

if not observation.exposure_ids:
    raise RuntimeError("The selected archive contains no parsed exposures")

print("Archive:", observation.archive_path)
print("Observation ID:", observation.observation_id)
print("Instrument:", observation.discovered.instrument.value)
print("Observation date:", observation.discovered.date)
print("Inventory members:", len(observation.members))
print("Parsed FITS frames:", len(observation.parsed_frames))
print("Malformed FITS frames:", len(observation.malformed_frames))
print("\nExposure groups:")
for exposure in observation.exposures:
    print(
        f"  {exposure.exposure_id}: {len(exposure.frames)} frame(s), "
        f"frame type(s)={exposure.frame_types}"
    )


## 4. Scientific exposure identity and classification

This table is the metadata checkpoint before detector processing. It reports
what the selected archive actually contains using the existing exposure-level
metadata. Missing fields remain visible as `—`; disagreements between member
headers are reported below the table rather than silently resolved.


In [ ]:
from html import escape


def _metadata_display_value(value):
    if value is None or value == "":
        return "—"
    if hasattr(value, "isoformat"):
        return value.isoformat(sep=" ")
    if isinstance(value, (tuple, list)):
        return ", ".join(_metadata_display_value(item) for item in value)
    return str(value)


def _header_value(metadata, *keys):
    for key in keys:
        value = metadata.header_values.get(key)
        if value not in (None, ""):
            return value
    return None


def exposure_identity_rows(obs):
    rows = []
    for exposure in obs.exposures:
        metadata = exposure.metadata
        classification = exposure.classification
        rows.append({
            "Exposure ID": exposure.exposure_id,
            "Frame type": metadata.frame_type or metadata.frame_types,
            "Frame class": metadata.frame_class,
            "Object (OBJECT)": metadata.object_name,
            "Program ID (QPROG)": metadata.program_id,
            "PROGRAM header": _header_value(metadata, "PROGRAM"),
            "Exposure time [s]": metadata.exposure_time_s,
            "Planned time [s]": metadata.planned_exposure_time_s,
            "Observation UTC": metadata.observation_time,
            "QRA": metadata.qra,
            "QDEC": metadata.qdec,
            "Telescope RA (TELRA)": _header_value(metadata, "TELRA"),
            "Telescope DEC (TELDEC)": _header_value(metadata, "TELDEC"),
            "Airmass": metadata.airmass,
            "Ambient temp [C]": metadata.ambient_temperature,
            "Humidity [%]": metadata.humidity,
            "Pressure": metadata.pressure,
            "QOBJECT": metadata.qobject,
            "Target": metadata.requested_target,
            "IFU slot": metadata.requested_ifuslot,
            "Observing mode": metadata.observing_mode,
            "Quick-look kind": classification.quicklook_kind,
            "Calibration source": classification.calibration_source,
            "Standard star": classification.standard_star,
            "Amplifiers": tuple(sorted(
                f"{identity.ifu_slot}{identity.amplifier}"
                for identity in exposure.physical_identities.values()
            )),
            "Frames": len(exposure.frames),
        })
    return rows


def display_exposure_identity(obs):
    print("Archive:", obs.archive_path)
    print("Instrument:", obs.discovered.instrument.value)
    print("Observation date:", obs.discovered.date)
    print("Exposure groups:", len(obs.exposures))

    rows = exposure_identity_rows(obs)
    if not rows:
        print("No parsed exposure metadata is available.")
        return

    columns = tuple(rows[0])
    header = "".join(
        f"<th style='border:1px solid #bbb;padding:4px 6px;text-align:left'>"
        f"{escape(column)}</th>"
        for column in columns
    )
    body = []
    for row in rows:
        cells = "".join(
            f"<td style='border:1px solid #bbb;padding:4px 6px'>"
            f"{escape(_metadata_display_value(row[column]))}</td>"
            for column in columns
        )
        body.append(f"<tr>{cells}</tr>")
    display(HTML(
        "<div style='overflow-x:auto'><table style='border-collapse:collapse'>"
        "<thead><tr>" + header + "</tr></thead>"
        "<tbody>" + "".join(body) + "</tbody></table></div>"
    ))

    for exposure in obs.exposures:
        disagreements = exposure.metadata.disagreements
        if disagreements:
            print(f"\nHeader disagreements for exposure {exposure.exposure_id}:")
            for disagreement in disagreements:
                print(
                    f"  {disagreement.field}: "
                    f"{_metadata_display_value(disagreement.values)}"
                )


display_exposure_identity(observation)


## 5. Select and load one amplifier frame

This remains the focused amplifier-level inspection path. It is useful for
checking one detector image, trace fit, extraction, and `SpatialQuicklook`
product in detail. The LRS2 presentation path below loads all eight amplifier
frames independently and then composes them into four channel products.

Set `EXPOSURE_ID` and `AMP_TOKEN` after inspecting the previous cells.


In [ ]:
if not observation.exposure_ids:
    raise RuntimeError("The selected archive contains no parsed exposures")
EXPOSURE_ID = observation.exposure_ids[0]
DEFAULT_AMP_TOKENS = {"lrs2": "056LL", "virus": "074LL"}
AMP_TOKEN = DEFAULT_AMP_TOKENS[INSTRUMENT]   # edit after inspecting the list

exposure = observation.exposure_for(EXPOSURE_ID)

def frame_amp_token(frame):
    ident = getattr(frame, "identity", None)
    return getattr(ident, "amp_token", None)

available = [(frame_amp_token(frame), getattr(frame, "identity", None), frame)
             for frame in exposure.frames]

print("Available amplifier tokens:")
print(sorted({token for token, _, _ in available if token is not None}))

selected_frame = None
if AMP_TOKEN is not None:
    matches = [frame for token, _, frame in available if token == AMP_TOKEN]
    if len(matches) != 1:
        raise ValueError(f"Expected one frame for {AMP_TOKEN}; found {len(matches)}")
    selected_frame = matches[0]
    loaded = observation.load_frame(selected_frame)
    print("loaded detector shape:", loaded.data.shape)
    print("identity:", loaded.identity)
    print("provenance:", loaded.provenance)
else:
    print("Set AMP_TOKEN to load a particular amplifier.")

## 6. Low-level detector evidence

Once `loaded` exists, inspect the minimal detector preparation independently of later presentation choices.

In [ ]:
if "loaded" in globals():
    detector_result = ql_detector.reduce_amplifier_array(
        loaded.data,
        dict(loaded.header),
    )
    detector = detector_result.get_array("oriented_detector_image")
    detector_variance = detector_result.get_array("detector_variance")
    print("Detector reduction kind:", detector_result.kind)
    print("Scalars:", detector_result.scalars)
    for name, value in detector_result.arrays.items():
        if value is not None:
            print(name, np.asarray(value).shape)
else:
    print("Load an amplifier in the previous cell first.")

## 7. Build or attach an amplifier-level topology

The numerical workflows consume an amplifier detector image plus amplifier-level fiber topology. The topology contains two distinct pieces of information:

- **detector trace**: where each fiber lies on the CCD;
- **physical IFU position**: where that same fiber lies in the IFU plane.

Those coordinate systems should remain separate.

This cell makes the current archive→topology handoff explicit: it resolves the physical identity from the selected frame, loads the dated trace reference and authoritative IFU positions, fits the dense detector trace, and constructs the small amplifier-level `FiberTopology` consumed by the workflow.

In [ ]:
for name in [
    "VirusTopologyLoader",
    "LRS2FiberPositionLoader",
]:
    show_signature(ql_topology, name)

show_signature(ql_workflows, "build_amplifier_topology")

if all(name in globals() for name in ("loaded", "detector_result", "selected_frame")):
    topology_result = ql_workflows.build_amplifier_topology(
        detector_result.get_array("oriented_detector_image"),
        exposure.identity_for(selected_frame),
        trace_root=TRACE_ROOT,
        at=DATE,
    )
    topology = topology_result.topology
    trace_result = topology_result.trace_result
    physical_identity = topology_result.physical_identity
    trace_provenance = topology_result.trace_provenance
    position_provenance = topology_result.position_provenance
    print("physical identity:", physical_identity)
    print("trace provenance:", trace_provenance)
    print("position provenance:", position_provenance)
    print("topology fibers:", len(topology.fiber_ids))
else:
    print("Load an amplifier and detector result before building topology.")


## 8. Run a quick-look workflow

Once `detector` and `topology` are available, this is the science-to-presentation boundary.

Use the already-reduced/oriented detector image if that is what the current workflow signature expects; the API inspection cell above is intentionally visible so this remains explicit.

In [ ]:
QUICKLOOK_KIND = "standard"   # "flat" or "standard"

# The previous cells provide detector, detector_variance, and topology.
workflow_variance = globals().get("detector_variance")

if "detector" in globals() and "topology" in globals():
    if QUICKLOOK_KIND == "flat":
        result = ql_workflows.run_ldls_flat_quicklook(
            detector,
            topology,
            detector_variance=workflow_variance,
            instrument=INSTRUMENT,
        )
    elif QUICKLOOK_KIND == "standard":
        result = ql_workflows.run_standard_star_quicklook(
            detector,
            topology,
            detector_variance=workflow_variance,
            instrument=INSTRUMENT,
        )
    else:
        raise ValueError("QUICKLOOK_KIND must be 'flat' or 'standard'")
    print(result)
else:
    print("Define `detector` and `topology` before running this cell.")

## 9. Result evidence inventory

Before deciding how a figure should look, inspect what the workflow already gives us. Presentation should consume this evidence rather than recomputing hidden scientific quantities.

## 9a. Build the LRS2 channel-level evidence products

For LRS2, each amplifier still follows the complete detector-to-fiber path
independently. The two amplifier products are composed only after central-
column collapse, when each fiber has a collapsed value and an authoritative
physical IFU position. This produces four 280-fiber channel products while
retaining the amplifier products and their lower-level evidence.


In [ ]:
if INSTRUMENT == "lrs2":
    if selected_frame is None:
        raise RuntimeError("Run the amplifier selection cell and load a frame first.")

    lrs2_run = ql_workflows.run_lrs2_channel_quicklooks(
        exposure,
        trace_root=TRACE_ROOT,
        at=DATE,
        frame_type=selected_frame.identity.frame_type,
        quicklook_kind=QUICKLOOK_KIND,
    )
    lrs2_amplifier_products = lrs2_run.amplifier_products
    lrs2_amplifier_evidence = lrs2_run.amplifier_evidence
    channel_products = lrs2_run.channels

    print("LRS2 amplifier products:", len(lrs2_amplifier_products))
    print("LRS2 channel products:")
    for channel_name, channel_product in channel_products.items():
        centroid = channel_product.measured_centroid
        measured = None if centroid is None else (centroid.x, centroid.y)
        print(
            f"  {channel_name}: {channel_product.fiber_values.size} fibers, "
            f"image={channel_product.image.shape}, measured={measured}"
        )
else:
    lrs2_amplifier_products = {}
    lrs2_amplifier_evidence = {}
    channel_products = {}
    print("VIRUS uses the existing selected-IFU/amplifier quick-look product.")


In [ ]:
def available_result_evidence(result):
    names = [
        "instrument",
        "channel",
        "amplifier_products",
        "fiber_values",
        "fiber_positions",
        "image",
        "spatial_x_coordinates",
        "spatial_y_coordinates",
        "spatial_support",
        "spatial_weight",
        "measured_centroid",
        "offset",
        "intended_fiducial",
        "collapse_columns",
        "collapse_statistic",
        "spatial_gaussian_fwhm_arcsec",
        "spatial_pixel_scale_arcsec",
        "extracted_spectra",
        "extraction_variance",
        "extraction_valid_fraction",
        "effective_aperture_width",
        "extraction_valid",
    ]
    evidence = {}
    for name in names:
        if hasattr(result, name):
            value = getattr(result, name)
            if isinstance(value, np.ndarray):
                evidence[name] = value.shape
            elif hasattr(value, "items"):
                evidence[name] = f"{len(value)} entries"
            else:
                evidence[name] = value
    return evidence

if "result" in globals():
    for key, value in available_result_evidence(result).items():
        print(f"{key:28s} {value}")
else:
    print("Run a quick-look workflow first.")

# Presentation laboratory

The cells below deliberately keep plots separate. This makes it easy to decide which pieces deserve to survive into an operator-facing diagnostic rather than committing prematurely to a dashboard layout.

Suggested evidence hierarchy:

1. reconstructed spatial image;
2. physical fiber samples;
3. intended and measured standard-star locations;
4. spatial support;
5. extraction/trace evidence when diagnosing something odd.

In [ ]:
def plot_lrs2_channels(
    channels,
    *,
    title=None,
    percentiles=(2.0, 98.0),
    show_fibers=True,
    show_fiducial=True,
    show_centroid=True,
):
    """Render the exploratory UV/Orange/Red/Far-Red 2x2 layout."""

    order = ("UV", "Orange", "Red", "Far-Red")
    images = [np.asarray(channels[name].image, dtype=float) for name in order]
    finite = np.concatenate([
        image[np.isfinite(image)] for image in images if np.any(np.isfinite(image))
    ]) if any(np.any(np.isfinite(image)) for image in images) else np.array([])
    vmin, vmax = ql_visualization.finite_limits(finite, percentiles)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    for ax, channel_name in zip(axes.flat, order):
        channel = channels[channel_name]
        image = np.asarray(channel.image, dtype=float)
        x = np.asarray(channel.spatial_x_coordinates, dtype=float)
        y = np.asarray(channel.spatial_y_coordinates, dtype=float)
        kwargs = {"origin": "lower", "aspect": "equal"}
        if vmin is not None:
            kwargs.update(vmin=vmin, vmax=vmax)
        im = ax.imshow(
            image,
            extent=ql_visualization.grid_extent(x, y),
            **kwargs,
        )
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Collapsed signal")

        if show_fibers:
            positions = np.asarray(channel.fiber_positions, dtype=float)
            ax.scatter(
                positions[:, 0], positions[:, 1],
                facecolors="none", s=10, linewidths=0.35,
            )
        if show_fiducial and channel.intended_fiducial is not None:
            fx, fy = channel.intended_fiducial
            ax.scatter([fx], [fy], marker="x", s=90, linewidths=1.8, label="Intended")
        if show_centroid and channel.measured_centroid is not None:
            cx, cy = channel.measured_centroid.x, channel.measured_centroid.y
            if np.all(np.isfinite([cx, cy])):
                ax.scatter([cx], [cy], marker="+", s=110, linewidths=1.8, label="Measured")
        if ax.get_legend_handles_labels()[0]:
            ax.legend(loc="best", fontsize="small")
        ax.set_xlabel("IFU x [arcsec]")
        ax.set_ylabel("IFU y [arcsec]")
        ax.set_title(channel_name)

    if title:
        fig.suptitle(title)
    return fig


In [ ]:
# Reusable plotting implementation lives in hetquicklook.visualization.
plot_fiber_values = ql_visualization.plot_fiber_values


In [ ]:
# Reusable plotting implementation lives in hetquicklook.visualization.
plot_spatial_support = ql_visualization.plot_spatial_support


## 10. First presentation pass

The percentile stretch below affects **display only**. It does not alter the stored fiber values or reconstructed image.

For flat health checks, compare the reconstructed image with the fiber-level view: the latter is often the quickest way to tell whether a dark patch is supported by actual low fibers rather than by interpolation.

For standard stars, keep the `(0,0)` intended marker and measured location visible, but do not turn the separation into a quality grade.

In [ ]:
DISPLAY_PERCENTILES = (2.0, 98.0)

if INSTRUMENT == "lrs2" and channel_products:
    fig_spatial = plot_lrs2_channels(
        channel_products,
        title=f"{DATE} LRS2 {QUICKLOOK_KIND} channel quick look",
        percentiles=DISPLAY_PERCENTILES,
        show_fibers=True,
        show_fiducial=(QUICKLOOK_KIND == "standard"),
        show_centroid=(QUICKLOOK_KIND == "standard"),
    )
    plt.show()
else:
    fig_spatial = ql_visualization.plot_spatial_image(
        result,
        percentiles=DISPLAY_PERCENTILES,
        show_fibers=True,
        show_fiducial=(QUICKLOOK_KIND == "standard"),
        show_centroid=(QUICKLOOK_KIND == "standard"),
    )
    plt.show()

# Keep an amplifier-level view available for detailed inspection.
if "result" in globals():
    fig_fibers = ql_visualization.plot_fiber_values(
        result,
        title=f"{AMP_TOKEN} amplifier fiber evidence",
        percentiles=DISPLAY_PERCENTILES,
    )
    plt.show()

    fig_support = ql_visualization.plot_spatial_support(
        result,
        title=f"{AMP_TOKEN} amplifier spatial support",
    )
    plt.show()


## 11. Presentation experiments

This is where we should iterate together. Useful questions include:

- For flats, should the primary view emphasize absolute collapsed signal or relative fiber throughput?
- Should the fiber-center overlay always be visible, or only in a diagnostic view?
- Is a linear percentile stretch sufficient, or does a second normalized view reveal obstruction structure better?
- For standard stars, does the weighted centroid visually land where a human would put it?
- Which metadata belongs in the title versus a small evidence block?
- For VIRUS, do we want an IFU-detail product plus a separate whole-focal-plane overview?

Change **display choices here**, not the reduction algorithms.

In [ ]:
# Presentation knobs — intentionally separate from science defaults.
SHOW_FIBERS = True
SHOW_SUPPORT = False
DISPLAY_PERCENTILES = (1.0, 99.0)

if INSTRUMENT == "lrs2" and channel_products:
    fig = plot_lrs2_channels(
        channel_products,
        title=f"{DATE} LRS2 {QUICKLOOK_KIND} channel experiment",
        percentiles=DISPLAY_PERCENTILES,
        show_fibers=SHOW_FIBERS,
        show_fiducial=(QUICKLOOK_KIND == "standard"),
        show_centroid=(QUICKLOOK_KIND == "standard"),
    )
    plt.show()
else:
    fig = ql_visualization.plot_spatial_image(
        result,
        percentiles=DISPLAY_PERCENTILES,
        show_fibers=SHOW_FIBERS,
        show_fiducial=(QUICKLOOK_KIND == "standard"),
        show_centroid=(QUICKLOOK_KIND == "standard"),
    )
    plt.show()

if SHOW_SUPPORT and "result" in globals():
    fig = ql_visualization.plot_spatial_support(result)
    plt.show()


## 12. Save candidate presentation figures

Keep exported figures separate from the scientific result. This lets us iterate on presentation without changing numerical outputs.

In [ ]:
OUTPUT_DIR = Path("notebook_output")
OUTPUT_DIR.mkdir(exist_ok=True)

def save_candidate(fig, name, *, dpi=160):
    path = OUTPUT_DIR / name
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(path.resolve())
    return path

# Example:
# save_candidate(fig_spatial, f"{DATE}_{INSTRUMENT}_{QUICKLOOK_KIND}_spatial.png")

# Next experiments

A useful sequence for the next interactive session:

1. Run one **LRS2 flat** and decide whether the four channel images + fiber samples make obstruction evidence obvious.
2. Run one **LRS2 standard** and judge the `(0,0)` versus measured channel positions.
3. Inspect one underlying LRS2 amplifier when a channel view looks unusual.
4. Repeat on one **VIRUS IFU**.
5. Only after the single-IFU presentation is satisfactory, design the **whole-VIRUS focal-plane overview**.
6. Promote only presentation choices that prove useful on real data into `visualization.py`.

The notebook should stay exploratory. Once a plot becomes operationally convincing, move the smallest reusable plotting logic into the package and leave the experiment history here.